In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/preprocessed_customer_data.csv")
df.head()


,customer_id,total_spent,total_orders,last_purchase,recency,avg_order_value
0,00012a2ce6f8dcda20d059ce98491703,1950.58,1,2017-11-14 16:08:26,287,1950.58
1,000161a058600d5901f007fab4c27140,1145.97,1,2017-07-16 09:40:32,409,1145.97
2,0001fd6190edaaf884bcaf3d49edf079,3322.14,1,2017-02-28 11:06:43,547,3322.14
3,0002414f95344307404f0ace7a26f1d5,3048.95,1,2017-08-16 13:09:20,378,3048.95
4,000379cdec625522490c315e70c7a9fb,1819.17,1,2018-04-02 13:42:17,149,1819.17


In [2]:
# Customer Segmentation Based on Spending
df['spending_segment'] = pd.qcut(
    df['total_spent'],
    q=3,
    labels=['Low', 'Medium', 'High']
)


In [ ]:
# RFM (Recency, Frequency, Monetary) Scoring
#  Recency (lower recency = better)
df['R'] = pd.qcut(df['recency'], 4, labels=[4, 3, 2, 1])

#  Frequency (rank-based to handle low variance)
df['F'] = pd.qcut(
    df['total_orders'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
)

#  Monetary
df['M'] = pd.qcut(df['total_spent'], 4, labels=[1, 2, 3, 4])

# RFM Score
df['RFM_Score'] = (
    df['R'].astype(str) +
    df['F'].astype(str) +
    df['M'].astype(str)
)

df[['R','F','M','RFM_Score']].head()



,R,F,M,RFM_Score
0,2,1,3,213
1,1,1,2,112
2,1,1,4,114
3,1,1,4,114
4,3,1,3,313


In [ ]:
# Customer Lifetime Value 
df['CLV'] = (
    df['avg_order_value'] *
    df['total_orders'] *
    (1 / (df['recency'] + 1))
)


In [6]:
# Churn Risk Estimation
df['churn_risk'] = (
    (df['recency'] / df['recency'].max()) * 0.6 +
    (1 - df['total_orders'] / df['total_orders'].max()) * 0.4
)


In [7]:
# Normalized Behavioral Features
df['norm_spent'] = df['total_spent'] / df['total_spent'].max()
df['norm_orders'] = df['total_orders'] / df['total_orders'].max()
df['norm_recency'] = 1 - (df['recency'] / df['recency'].max())


In [8]:
# Future Spending Estimation (Target Variable)
np.random.seed(42)

df['future_spending'] = (
    df['avg_order_value'] *
    (1 + df['norm_orders']) *
    (1 + df['norm_recency']) *
    np.random.uniform(0.8, 1.2, len(df))
)


In [9]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88901 entries, 0 to 88900
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   customer_id       88901 non-null  object  
 1   total_spent       88901 non-null  float64 
 2   total_orders      88901 non-null  int64   
 3   last_purchase     88901 non-null  object  
 4   recency           88901 non-null  int64   
 5   avg_order_value   88901 non-null  float64 
 6   spending_segment  88901 non-null  category
 7   R                 88901 non-null  category
 8   F                 88901 non-null  category
 9   M                 88901 non-null  category
 10  RFM_Score         88901 non-null  object  
 11  CLV               88901 non-null  float64 
 12  churn_risk        88901 non-null  float64 
 13  norm_spent        88901 non-null  float64 
 14  norm_orders       88901 non-null  float64 
 15  norm_recency      88901 non-null  float64 
 16  future_spending   8890

,customer_id,total_spent,total_orders,last_purchase,recency,avg_order_value,spending_segment,R,F,M,RFM_Score,CLV,churn_risk,norm_spent,norm_orders,norm_recency,future_spending
0,00012a2ce6f8dcda20d059ce98491703,1950.58,1,2017-11-14 16:08:26,287,1950.58,Medium,2,1,3,213,6.772847,0.247770,0.329741,1.0,0.587050,5880.631600
1,000161a058600d5901f007fab4c27140,1145.97,1,2017-07-16 09:40:32,409,1145.97,Low,1,1,2,112,2.795049,0.353094,0.193724,1.0,0.411511,3818.340031
2,0001fd6190edaaf884bcaf3d49edf079,3322.14,1,2017-02-28 11:06:43,547,3322.14,High,1,1,4,114,6.062299,0.472230,0.561600,1.0,0.212950,8807.049135
3,0002414f95344307404f0ace7a26f1d5,3048.95,1,2017-08-16 13:09:20,378,3048.95,High,1,1,4,114,8.044723,0.326331,0.515418,1.0,0.456115,9229.649431
4,000379cdec625522490c315e70c7a9fb,1819.17,1,2018-04-02 13:42:17,149,1819.17,Medium,3,1,3,313,12.127800,0.128633,0.307527,1.0,0.785612,5602.769563


In [10]:
df.isnull().sum()


customer_id         0
total_spent         0
total_orders        0
last_purchase       0
recency             0
avg_order_value     0
spending_segment    0
R                   0
F                   0
M                   0
RFM_Score           0
CLV                 0
churn_risk          0
norm_spent          0
norm_orders         0
norm_recency        0
future_spending     0
dtype: int64

In [11]:
df[['churn_risk','norm_spent','norm_orders','norm_recency']].describe()


,churn_risk,norm_spent,norm_orders,norm_recency
count,88901.000000,88901.000000,88901.0,88901.000000
mean,0.206303,0.330629,1.0,0.656162
std,0.131680,0.207354,0.0,0.219466
min,0.000000,0.027560,1.0,0.000000
25%,0.100144,0.168463,1.0,0.497842
50%,0.189928,0.279047,1.0,0.683453
75%,0.301295,0.446734,1.0,0.833094
max,0.600000,1.000000,1.0,1.000000


In [12]:
df['future_spending'].describe()


count    88901.000000
mean      6486.490954
std       4291.787865
min        414.805334
25%       3196.952261
50%       5352.901105
75%       8745.545776
max      27676.525048
Name: future_spending, dtype: float64

In [13]:
df.dtypes


customer_id           object
total_spent          float64
total_orders           int64
last_purchase         object
recency                int64
avg_order_value      float64
spending_segment    category
R                   category
F                   category
M                   category
RFM_Score             object
CLV                  float64
churn_risk           float64
norm_spent           float64
norm_orders          float64
norm_recency         float64
future_spending      float64
dtype: object

In [14]:
df.to_csv("../data/final_customer_data.csv", index=False)


In [15]:
model_features = [
    'total_spent',
    'total_orders',
    'avg_order_value',
    'recency',
    'CLV',
    'churn_risk',
    'norm_spent',
    'norm_orders',
    'norm_recency',
    'future_spending'
]

df[model_features].to_csv(
    "../data/model_ready_data.csv",
    index=False
)
